# 🛂 CodePassport AI — Colab Training Notebook

**Fine-tunes CodeT5-base with LoRA to generate structured Developer Passports from Python functions.**

### Steps:
1. Install dependencies (cell 1)
2. Upload your processed dataset (cell 2)
3. Verify GPU (cell 3)
4. Run fine-tuning (cell 4)
5. Test inference (cell 5)
6. Download model (cell 6)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Install Dependencies
# Run this FIRST. It pins versions to avoid conflicts on Colab.
# Expected time: 3-5 minutes
# ════════════════════════════════════════════════════════════

# Uninstall conflicting pre-installed versions
!pip uninstall -y transformers tokenizers accelerate peft -q

# Install pinned versions
!pip install -q \
    torch>=2.1.0 \
    transformers==4.40.2 \
    peft==0.10.0 \
    accelerate==0.29.3 \
    bitsandbytes==0.43.1 \
    datasets==2.19.0 \
    tokenizers==0.19.1 \
    sentencepiece==0.2.0 \
    nltk==3.8.1 \
    rouge-score==0.1.2 \
    tqdm==4.66.2

print('\n✅ All packages installed!')

# Verify key versions
import transformers, peft, torch
print(f'   torch        : {torch.__version__}')
print(f'   transformers : {transformers.__version__}')
print(f'   peft         : {peft.__version__}')
print(f'   CUDA         : {torch.cuda.is_available()}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Upload Dataset Files
# Upload your processed JSONL files from data/processed/
# ════════════════════════════════════════════════════════════

from google.colab import files
import os

os.makedirs('data/processed', exist_ok=True)
os.makedirs('models/codepassport-lora', exist_ok=True)

print('📁 Upload your train.jsonl, val.jsonl, test.jsonl files:')
uploaded = files.upload()

# Move files to correct location
for fname in uploaded.keys():
    dest = f'data/processed/{fname}'
    os.rename(fname, dest)
    size_mb = os.path.getsize(dest) / 1e6
    print(f'   ✅ {fname} → {dest}  ({size_mb:.1f} MB)')

# Verify
import json
for split in ['train', 'val', 'test']:
    path = f'data/processed/{split}.jsonl'
    if os.path.exists(path):
        with open(path) as f:
            count = sum(1 for line in f if line.strip())
        print(f'   {split}: {count:,} records')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — Verify GPU
# Make sure you have a T4 or better (Runtime > Change runtime type > T4)
# ════════════════════════════════════════════════════════════

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        '❌ No GPU detected! '
        'Go to Runtime > Change runtime type > Hardware accelerator = GPU (T4)'
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'✅ GPU detected: {gpu_name}')
print(f'   VRAM: {vram_gb:.1f} GB')
if vram_gb < 10:
    print('⚠️  Less than 10 GB VRAM. Reduce BATCH_SIZE to 4 in the training cell.')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Fine-Tune CodeT5 with LoRA
# Expected time: ~45-90 minutes for 3 epochs on T4
# ════════════════════════════════════════════════════════════

import os, json, math, torch
from transformers import (
    AutoTokenizer,
    T5ForConditionalGeneration,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    EarlyStoppingCallback,
)
from peft import get_peft_model, LoraConfig, TaskType
from datasets import Dataset

# ── CONFIG (adjust if OOM) ──────────────────────────────────
MODEL_NAME      = 'Salesforce/codet5-base'
OUTPUT_DIR      = 'models/codepassport-lora'
MAX_INPUT_LEN   = 512
MAX_TARGET_LEN  = 256
BATCH_SIZE      = 8     # ← reduce to 4 if you get OOM errors
GRAD_ACCUM      = 4
EPOCHS          = 3
LR              = 3e-4
LORA_R          = 16
LORA_ALPHA      = 32
DEVICE          = 'cuda'

print(f'🔤 Loading tokenizer: {MODEL_NAME}')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ── Helper to load JSONL ──────────────────────────────────
def read_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

# ── Tokenize split ────────────────────────────────────────
def tokenize_records(records, split_name):
    print(f'⚙️  Tokenizing {split_name} ({len(records):,} samples)...')
    prompts = [r['prompt'] for r in records]
    targets = [r['target'] for r in records]
    tok = tokenizer(
        prompts, text_target=targets,
        max_length=MAX_INPUT_LEN, max_target_length=MAX_TARGET_LEN,
        padding=False, truncation=True,
    )
    labels_clean = [
        [t if t != tokenizer.pad_token_id else -100 for t in row]
        for row in tok['labels']
    ]
    return Dataset.from_dict({
        'input_ids': tok['input_ids'],
        'attention_mask': tok['attention_mask'],
        'labels': labels_clean,
    })

train_ds = tokenize_records(read_jsonl('data/processed/train.jsonl'), 'train')
val_ds   = tokenize_records(read_jsonl('data/processed/val.jsonl'),   'val')
print(f'✅ Train: {len(train_ds):,} | Val: {len(val_ds):,}')

# ── Load model ───────────────────────────────────────────
print(f'\n🤖 Loading base model: {MODEL_NAME}')
model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16
)

# ── Apply LoRA ───────────────────────────────────────────
print('🎯 Applying LoRA adapters...')
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=['q', 'v'],
    lora_dropout=0.05,
    bias='none',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ── Training args ────────────────────────────────────────
steps_per_epoch = math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM))
eval_steps = max(steps_per_epoch // 4, 50)

training_args = Seq2SeqTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_ratio=0.05,
    fp16=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    evaluation_strategy='steps',
    eval_steps=eval_steps,
    save_strategy='steps',
    save_steps=eval_steps,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    save_total_limit=2,
    report_to='none',
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model,
    padding=True, label_pad_token_id=-100
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f'\n🚂 Training started! Epochs={EPOCHS}, EffectiveBatch={BATCH_SIZE*GRAD_ACCUM}')
trainer.train()

# ── Save ─────────────────────────────────────────────────
print(f'\n💾 Saving model → {OUTPUT_DIR}')
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print('\n✅ Training complete and model saved!')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Quick Inference Test
# Test the fine-tuned model directly in Colab
# ════════════════════════════════════════════════════════════

import torch
from transformers import AutoTokenizer, T5ForConditionalGeneration
from peft import PeftModel, PeftConfig

MODEL_PATH = 'models/codepassport-lora'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Load
print('📦 Loading fine-tuned model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
try:
    config   = PeftConfig.from_pretrained(MODEL_PATH)
    base     = T5ForConditionalGeneration.from_pretrained(config.base_model_name_or_path)
    model    = PeftModel.from_pretrained(base, MODEL_PATH).merge_and_unload()
except Exception:
    model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)
model = model.to(DEVICE).eval()
print('✅ Model loaded!')

# Test function
test_code = '''
def binary_search(arr, target):
    left, right = 0, len(arr) - 1
    while left <= right:
        mid = (left + right) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1
'''

prompt = (
    'Generate a structured developer passport for the following Python function.\n'
    'Include these sections: DOCSTRING, PURPOSE, BEHAVIOR SUMMARY, '
    'INPUTS / OUTPUTS, ASSUMPTIONS, EDGE CASES, DEVELOPER NOTE.\n\n'
    f'### Python Function:\n{test_code.strip()}\n\n'
    '### Developer Passport:\n'
)

inputs = tokenizer(prompt, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)

with torch.no_grad():
    outputs = model.generate(
        input_ids=inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=300,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3,
    )

result = tokenizer.decode(outputs[0], skip_special_tokens=True)

print('\n' + '═'*60)
print('  🛂 GENERATED DEVELOPER PASSPORT')
print('═'*60)
print(result)
print('═'*60)

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — Download Trained Model
# Zips and downloads the model folder to your PC.
# ════════════════════════════════════════════════════════════

import shutil
from google.colab import files

print('📦 Zipping model folder...')
shutil.make_archive('codepassport-lora', 'zip', 'models/codepassport-lora')

zip_size_mb = os.path.getsize('codepassport-lora.zip') / 1e6
print(f'   Archive size: {zip_size_mb:.1f} MB')
print('⬇️  Downloading...')
files.download('codepassport-lora.zip')
print('✅ Done! Unzip to models/codepassport-lora/ on your local PC.')